In [74]:
import pandas as pd
import os
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.metrics import brier_score_loss

In [75]:
'''
Function to merge two dataframes
param df1: first dataframe
param df2: second dataframe
return: a single dataframe of the merged dataframes 
'''
def mergeDataframes(df1, df2):
    df = pd.concat([df1, df2])
    return df


'''
Function to merge tournament data for mens and womens and crop the data to match reg detail
params df1: dataframe of mens tournament data
params df2: dataframe of womens tournament data
return: dataframe of combined and cropped tournament data
'''
def mergeTournamentData(df1, df2):
    df1 = df1[df1['Season'] >= 2003].reset_index(drop=True)
    df2 = df2[df2['Season'] >= 2010].reset_index(drop=True)
    
    df = pd.concat([df1, df2])
    return df


'''
Function to split regular season detailed results into dataframes focused on outcome for one team
param df: regular season data
return: a dataframe where each team from a single row in reg data has its own row
'''
def regularDetailsFocus(df):
    RegWinners = pd.DataFrame()
    RegLossers = pd.DataFrame()

    # Establish new columns for that includes stats for one team
    columns = ['Season', 'TeamID', 'DayNum', 'Score', 'OppScore',
               'NumOT', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
               'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF', 'OppFGM', 'OppFGA',
               'OppFGM3', 'OppFGA3', 'OppFTM', 'OppFTA', 'OppOR', 'OppDR', 'OppAst', 'OppTO',
               'OppStl', 'OppBlk', 'OppPF']

    # Split winners from regular season
    RegWinners[columns] = df[['Season', 'WTeamID', 'DayNum', 'WScore', 'LScore',
                              'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA',
                              'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA',
                              'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO',
                              'LStl', 'LBlk', 'LPF']]

    # Add wins and losses columns
    RegWinners['Win'] = 1
    RegWinners['Loss'] = 0

    # Split lossers from regular season
    RegLossers[columns] = df[['Season', 'LTeamID', 'DayNum', 'LScore', 'WScore',
                               'NumOT', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA',
                               'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF', 'WFGM', 'WFGA',
                               'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO',
                               'WStl', 'WBlk', 'WPF']]

    # Add wins and losses columns
    RegLossers['Win'] = 0
    RegLossers['Loss'] = 1

    # Combine all games into one dataframe
    AllRegDetail = pd.concat([RegWinners, RegLossers])
    
    return AllRegDetail


'''
Function to clean seed column
param seeds: Dataframe of historical seeds with column 'Seed'
return: A dataframe with the 'Seed' column converted to int
'''

def cleanSeed(seeds):
    seeds['Seed'] = seeds['Seed'].str.extract(r'(\d+)').astype(int)
    return seeds


'''
Function to join two Dataframes on 'TeamID'
param seeds: Dataframe of historical seeds with column 'Seed'
param tourny: Dataframe of compact tournament data with columns 'WTeamID' and 'LTeamID'
return: a single dataframe of the joined Dataframes
'''

def joinSeeds(seeds, tourny):
    seeds = seeds.set_index(['Season','TeamID'])

    tourny = tourny.join(
        seeds.rename(columns={'Seed':'WSeed'}),
        on=['Season','WTeamID']
    )

    tourny = tourny.join(
        seeds.rename(columns={'Seed':'LSeed'}),
        on=['Season','LTeamID']
    )
    return tourny


'''
Function to create the features from the regular season data at a single-game level
param df: dataframe of regular season data
return: dataframe of input features at a single-game level
'''
def createSingleFeatures(df):
    RegSeasonFeatures = pd.DataFrame()
    RegSeasonFeatures[['Season', 'TeamID', 'DayNum']] = df[['Season', 'TeamID', 'DayNum']]

    RegSeasonFeatures['PointRatio'] = df['Score'] / df['OppScore']  # Points ratio
    RegSeasonFeatures['MOV'] = df['Score'] - df['OppScore']  # Margin of victory
    RegSeasonFeatures['TORatio'] = df['TO'] / df['OppTO']  # Turnover ratio
    RegSeasonFeatures['FGM%'] = df['FGM'] / df['FGA']  # Scoring efficiency
    RegSeasonFeatures['FG3%M'] = df['FGM3'] / df['FGA3']  # 3-Point efficiency
    RegSeasonFeatures['FGA3%'] = df['FGA3'] / df['FGA']  # 3-Point attempt rate
    RegSeasonFeatures['FTM%'] = df['FTM'] / df['FTA']  # Free throw makes %
    RegSeasonFeatures['OppFTM%'] = df['OppFTM'] / df['OppFTA']  # Opponent free throw makes %
    RegSeasonFeatures['FTR'] = df['FTA'] / df['FGA']  # Free throw attempt rate
    RegSeasonFeatures['OppFTR'] = df['OppFTA'] / df['OppFGA']  # Opponent free throw attempt rate
    RegSeasonFeatures['ORRatio'] = df['OR'] / (df['OR'] + df['OppDR'])  # Offensive rebound ratio
    RegSeasonFeatures['DRRatio'] = df['DR'] / (df['DR'] + df['OppOR'])  # Defensive rebound ratio
    
    # New for 2026 (More advanced!)
    RegSeasonFeatures['NumPos'] = df['FGA'] - df['OR'] + df['TO'] + (0.44 * df['FTA'])  # ROUGH estimation of positions
    RegSeasonFeatures['OffEff'] = df['Score'] / RegSeasonFeatures['NumPos']  # Offensive efficiency
    RegSeasonFeatures['DefEff'] = df['OppScore'] / RegSeasonFeatures['NumPos']  # Defensive efficiency
    RegSeasonFeatures['NetEff'] = RegSeasonFeatures['OffEff'] - RegSeasonFeatures['DefEff']  # Net efficiency
    RegSeasonFeatures['TO%'] = df['TO'] / RegSeasonFeatures['NumPos']  # Turnover %
    RegSeasonFeatures['Ast%'] = df['Ast'] / df['FGM']  # Assist percentage
    RegSeasonFeatures['AstTORatio'] = df['Ast'] / df['TO']  # Assist to turnover ratio
    RegSeasonFeatures['AstRatio'] = df['Ast'] / df['OppAst']  # Assist ratio
    RegSeasonFeatures['OR%'] = df['OR'] / (df['FGA'] - df['FGM'])  # Offensive rebound %
    RegSeasonFeatures['DR%'] = df['DR'] / (df['OppFGA'] - df['OppFGM'])  # Defensive rebound %
    RegSeasonFeatures['EffFG%'] = (df['FGM'] + (0.5 * df['FGM3'])) / df['FGA']  # Effective field goal %
    RegSeasonFeatures['TS%'] = df['Score'] / (2 * (df['FGA'] + (0.44 * df['FTA'])))  # True shot %
    
    return RegSeasonFeatures

'''
Function to create the final features for input to the model
param df1: dataframe of single-game features
param df2: dataframe of regular season boxscore with columns ['Win', 'Loss']
return: dataframe of final input features
'''
def finalFeatures(df1, df2):
    # Season Average Features
    seasonAvg = df1.drop(columns='DayNum').groupby(['Season', 'TeamID']).mean().reset_index()
    
    # Get a win %
    winLoss = df2.groupby(['Season','TeamID'])[['Win', 'Loss']].sum().reset_index()
    winLoss['W/L'] = winLoss['Win'] / (winLoss['Win'] + winLoss['Loss'])  # Win/Loss ratio
    winLoss = winLoss.drop(['Win', 'Loss'], axis=1)
    
    # Merge season avg with win/loss
    finalSeasonAvg = pd.merge(seasonAvg, winLoss)
    
    # Rename columns appropriately
    finalSeasonAvg.columns = ['Season', 'TeamID'] + [f'Season_Avg_{col}' for col in finalSeasonAvg.columns if col not in ['Season', 'TeamID']]
    
    # Get the last 5 of the features
    lastFiveAvg = df1.sort_values(['Season', 'TeamID', 'DayNum']).groupby(['Season', 'TeamID']).tail(5).drop(columns='DayNum').groupby(['Season', 'TeamID']).mean().reset_index()
    lastFiveAvg.columns = ['Season', 'TeamID'] + [f'Last_5_Avg_{col}' for col in lastFiveAvg.columns if col not in ['Season', 'TeamID']]

    # Merge the season with rolling
    features = pd.merge(finalSeasonAvg, lastFiveAvg)
    
    return features

In [76]:
# Mens data import
mRegDetail = pd.read_csv('data/men/MRegularSeasonDetailedResults.csv')
mTournCompact = pd.read_csv('data/men/MNCAATourneyCompactResults.csv')
mTournSeeds = pd.read_csv('data/men/MNCAATourneySeeds.csv')
mNames = pd.read_csv('data/men/MTeamSpellings.csv')

# Womens data import
wRegDetail = pd.read_csv('data/women/WRegularSeasonDetailedResults.csv')
wTournCompact = pd.read_csv('data/women/WNCAATourneyCompactResults.csv')
wTournSeeds = pd.read_csv('data/women/WNCAATourneySeeds.csv')
wNames = pd.read_csv('data/women/WTeamSpellings.csv') 

# Clean and merge seeds with tournament results
mCleanSeeds = cleanSeed(mTournSeeds.copy())
wCleanSeeds = cleanSeed(wTournSeeds.copy())
mFullTourn = joinSeeds(mCleanSeeds, mTournCompact)
wFullTourn = joinSeeds(wCleanSeeds, wTournCompact)

# Combined data
regDetail = mergeDataframes(mRegDetail, wRegDetail)
compactTourn = mergeTournamentData(mFullTourn, wFullTourn)
names = mergeDataframes(mNames, wNames)

# Split regular season detailed results into dataframes focused on outcome for one team
AllRegDetail = regularDetailsFocus(regDetail)

# Create single-game features
singleFeatures = createSingleFeatures(AllRegDetail)

features = finalFeatures(singleFeatures, AllRegDetail)

In [77]:
features

,Season,TeamID,Season_Avg_PointRatio,Season_Avg_MOV,Season_Avg_TORatio,Season_Avg_FGM%,Season_Avg_FG3%M,Season_Avg_FGA3%,Season_Avg_FTM%,Season_Avg_OppFTM%,...,Last_5_Avg_DefEff,Last_5_Avg_NetEff,Last_5_Avg_TO%,Last_5_Avg_Ast%,Last_5_Avg_AstTORatio,Last_5_Avg_AstRatio,Last_5_Avg_OR%,Last_5_Avg_DR%,Last_5_Avg_EffFG%,Last_5_Avg_TS%
0,2003,1102,1.046075,0.250000,0.927235,0.486149,0.367637,0.518525,0.642402,0.719462,...,1.068804,-0.070695,0.206511,0.514955,0.874928,1.391667,0.176438,0.902105,0.538749,0.580584
1,2003,1103,1.012673,0.629630,0.884077,0.487294,0.331990,0.288910,0.735271,0.735661,...,1.081408,0.055210,0.206911,0.593730,1.251685,1.044561,0.333719,0.647479,0.587461,0.616565
2,2003,1104,1.080004,4.285714,1.033545,0.419676,0.325442,0.346570,0.705168,0.712597,...,1.000801,0.071834,0.210314,0.432000,0.938496,0.910823,0.384643,0.806401,0.519859,0.551370
3,2003,1105,0.957745,-4.884615,1.049965,0.396204,0.359630,0.338957,0.709598,0.669716,...,1.010096,-0.007405,0.196536,0.683143,1.299145,1.123953,0.350720,0.719341,0.484071,0.519153
4,2003,1106,1.027887,-0.142857,1.207946,0.425530,0.350196,0.322344,0.623158,0.711733,...,0.965899,-0.033005,0.234829,0.458607,0.630000,1.153464,0.295883,0.760481,0.486013,0.518412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14306,2026,3477,0.953849,-5.192308,1.170476,0.381132,0.313023,0.342042,0.742978,0.740678,...,1.053389,-0.141375,0.111594,0.511124,1.626984,0.687634,0.246076,0.600244,0.384046,0.438156
14307,2026,3478,0.846574,-14.448276,1.430331,0.373077,0.313402,0.396489,0.769490,0.753584,...,1.090258,-0.008041,0.095815,0.534837,3.192857,1.036410,0.280211,0.605116,0.454274,0.515542
14308,2026,3479,0.921181,-9.785714,1.755088,0.384967,0.332310,0.497672,0.750460,0.736666,...,0.878748,0.157637,0.142134,0.660379,1.771429,1.148810,0.229718,0.648197,0.476348,0.527451
14309,2026,3480,1.049563,0.777778,1.423562,0.422148,0.329954,0.430180,0.703031,0.694238,...,1.152531,-0.024116,0.109976,0.349329,1.451414,0.804444,0.284926,0.639378,0.520728,0.549548
